# 🏆 Notebook 04 — Model 4: RoBERTa NER + Final Comparison
**Technology**: `roberta-base` — Heavyweight Transformer (HuggingFace 🤗)  
**Environment**: Google Colab T4 GPU  
**Role in project**: State-of-the-art model, achieves highest F1. Final comparison of all 4 models.

---
### 📁 Setup Instructions
1. Upload to Colab (same files as Notebook 03):
   - `data/train.json`, `data/test.json`, `data/labels.json`
   - `results/model1_results.json`, `results/model2_results.json`, `results/model3_results.json`
2. Set Runtime → **T4 GPU**


In [ ]:
!pip install transformers datasets seqeval accelerate -q

In [ ]:
import json, os, random
import numpy as np
import torch
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset
import evaluate

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
DATA_DIR = Path("data")
train_raw     = json.load(open(DATA_DIR / "train.json", encoding="utf-8"))
test_raw      = json.load(open(DATA_DIR / "test.json",  encoding="utf-8"))
ENTITY_LABELS = json.load(open(DATA_DIR / "labels.json"))

bio_labels = ["O"] + [f"B-{l}" for l in ENTITY_LABELS] + [f"I-{l}" for l in ENTITY_LABELS]
bio_labels = ["O"]
for label in ENTITY_LABELS:
    bio_labels.append(f"B-{label}")
    bio_labels.append(f"I-{label}")

label2id = {l: i for i, l in enumerate(bio_labels)}
id2label = {i: l for l, i in label2id.items()}
print(f"✅ Loaded data | Labels: {len(bio_labels)} BIO tags")


## 1. Tokenization (RoBERTa uses BPE — different from DistilBERT)

In [ ]:
MODEL_CHECKPOINT = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, add_prefix_space=True)
# NOTE: add_prefix_space=True is important for RoBERTa's BPE tokenizer
# so that the first word of a sequence gets proper tokenization

def align_labels(text, entities, tokenizer, label2id, max_length=512):
    """Character-level → token-level BIO label alignment (works for both BERT & RoBERTa)."""
    char_label = {}
    for s, e, lbl in sorted(entities, key=lambda x: -(x[1]-x[0])):
        for i in range(int(s), int(e)):
            if i not in char_label:
                char_label[i] = f"B-{lbl}" if i == int(s) else f"I-{lbl}"

    encoding = tokenizer(text, truncation=True, max_length=max_length,
                         return_offsets_mapping=True, padding=False,
                         is_split_into_words=False)
    offsets  = encoding.pop("offset_mapping")

    token_labels = []
    for tok_s, tok_e in offsets:
        if tok_s == tok_e:
            token_labels.append(-100)
        else:
            raw_lbl = char_label.get(tok_s, "O")
            token_labels.append(label2id.get(raw_lbl, label2id["O"]))

    encoding["labels"] = token_labels
    return encoding

def build_dataset(raw_data):
    return Dataset.from_list([
        align_labels(text, ann["entities"], tokenizer, label2id)
        for text, ann in raw_data
    ])

print("Building datasets for RoBERTa...")
train_dataset = build_dataset(train_raw)
test_dataset  = build_dataset(test_raw)
print(f"✅ Train: {len(train_dataset)} | Test: {len(test_dataset)}")


## 2. Load RoBERTa Model & Train

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(bio_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model: {MODEL_CHECKPOINT}")
print(f"   Total params: {total_params:,} (~{total_params/1e6:.0f}M)")
print(f"   (vs DistilBERT: ~66M — RoBERTa is ~2× larger)")


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        p_seq, l_seq = [], []
        for p_id, l_id in zip(pred_seq, label_seq):
            if l_id != -100:
                p_seq.append(id2label[p_id])
                l_seq.append(id2label[l_id])
        true_preds.append(p_seq)
        true_labels.append(l_seq)
    r = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"precision": round(r["overall_precision"],4),
            "recall":    round(r["overall_recall"],4),
            "f1":        round(r["overall_f1"],4)}

training_args = TrainingArguments(
    output_dir          = "models/roberta-ner",
    num_train_epochs    = 10,
    per_device_train_batch_size = 8,   # Smaller batch — RoBERTa is larger
    per_device_eval_batch_size  = 8,
    learning_rate       = 2e-5,        # Slightly lower LR for larger model
    weight_decay        = 0.01,
    warmup_ratio        = 0.1,
    evaluation_strategy = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1",
    logging_steps       = 20,
    fp16                = torch.cuda.is_available(),
    report_to           = "none",
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=test_dataset,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("✅ Trainer ready. Starting training...")


In [ ]:
train_result = trainer.train()
print(f"✅ Training complete in {train_result.metrics['train_runtime']:.1f}s")


## 3. Evaluate RoBERTa

In [ ]:
predictions, labels_out, _ = trainer.predict(test_dataset)
preds = np.argmax(predictions, axis=2)

true_preds, true_labels = [], []
for pred_seq, label_seq in zip(preds, labels_out):
    p_seq, l_seq = [], []
    for p_id, l_id in zip(pred_seq, label_seq):
        if l_id != -100:
            p_seq.append(id2label[p_id])
            l_seq.append(id2label[l_id])
    true_preds.append(p_seq)
    true_labels.append(l_seq)

detailed = seqeval.compute(predictions=true_preds, references=true_labels)

print(f"{'Label':30s} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>6}")
print("-" * 55)
for ent_lbl in ENTITY_LABELS:
    if ent_lbl in detailed:
        m = detailed[ent_lbl]
        print(f"{ent_lbl:30s} {m['precision']:6.3f} {m['recall']:6.3f} {m['f1-score']:6.3f} {m['number']:6d}")
print("-" * 55)
print(f"{'OVERALL (micro)':30s} {detailed['overall_precision']:6.3f} "
      f"{detailed['overall_recall']:6.3f} {detailed['overall_f1']:6.3f}")

# Save
per_entity = {}
for ent_lbl in ENTITY_LABELS:
    if ent_lbl in detailed:
        m = detailed[ent_lbl]
        per_entity[ent_lbl] = {"precision": round(m['precision'],4),
                                "recall": round(m['recall'],4),
                                "f1": round(m['f1-score'],4)}
    else:
        per_entity[ent_lbl] = {"precision": 0, "recall": 0, "f1": 0}

output = {
    "model": "Model 4 - RoBERTa",
    "overall": {"precision": round(detailed['overall_precision'],4),
                "recall":    round(detailed['overall_recall'],4),
                "f1":        round(detailed['overall_f1'],4)},
    "per_entity": per_entity
}
json.dump(output, open("results/model4_results.json","w"), indent=2)
trainer.save_model("models/roberta-ner-final")
print(f"\n📌 RoBERTa Overall F1: {detailed['overall_f1']:.4f}")


## 4. 🏆 Final Comparison — All 4 Models

In [ ]:
# ── Load all results ─────────────────────────────────────────────
all_results = {}
for fname, name in [
    ("results/model1_results.json", "Rule-Based"),
    ("results/model2_results.json", "CNN (spaCy)"),
    ("results/model3_results.json", "DistilBERT"),
    ("results/model4_results.json", "RoBERTa"),
]:
    try:
        data = json.load(open(fname))
        all_results[name] = data
        ov = data["overall"]
        print(f"{name:15s} — P: {ov['precision']:.4f} | R: {ov['recall']:.4f} | F1: {ov['f1']:.4f}")
    except FileNotFoundError:
        print(f"⚠️  {fname} not found — run the previous notebooks first")

print("\n✅ Results loaded")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CHART 1: Overall Precision / Recall / F1 comparison
# ═══════════════════════════════════════════════════════════════
model_names = list(all_results.keys())
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Resume NER — Model Comparison", fontsize=15, fontweight='bold', y=1.02)

for ax, metric in zip(axes, ['precision', 'recall', 'f1']):
    vals = [all_results[m]['overall'][metric] for m in model_names]
    bars = ax.bar(model_names, vals, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha='center', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score", fontsize=11)
    ax.set_title(metric.capitalize(), fontsize=12, fontweight='bold')
    ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig("results/comparison_overall.png", bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CHART 2: Per-entity F1 heatmap-style comparison
# ═══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(16, 7))
x        = np.arange(len(ENTITY_LABELS))
n_models = len(all_results)
width    = 0.18

for i, (model_name, color) in enumerate(zip(model_names, colors)):
    f1_vals = [all_results[model_name]['per_entity'].get(lbl, {}).get('f1', 0)
               for lbl in ENTITY_LABELS]
    offset  = (i - n_models/2 + 0.5) * width
    bars = ax.bar(x + offset, f1_vals, width, label=model_name,
                  color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(ENTITY_LABELS, rotation=35, ha='right', fontsize=9)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title("Per-Entity F1 Score — All 4 Models", fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("results/comparison_per_entity.png", bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CHART 3: Technology Evolution — F1 progression
# ═══════════════════════════════════════════════════════════════
f1_progression = [all_results[m]['overall']['f1'] for m in model_names]
technologies   = ["EntityRuler\n(Pattern)", "tok2vec\n(CNN)", "Attention\n(DistilBERT)", 
                  "Deeper Attn\n(RoBERTa)"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(model_names)), f1_progression, 'o-', 
        color='steelblue', lw=2.5, markersize=10, markerfacecolor='white', markeredgewidth=2.5)

for i, (f1, name) in enumerate(zip(f1_progression, model_names)):
    ax.annotate(f"F1={f1:.3f}\n({name})", 
                xy=(i, f1), xytext=(0, 20), textcoords="offset points",
                ha='center', fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))

ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(technologies, fontsize=10)
ax.set_ylabel("Macro F1 Score", fontsize=12)
ax.set_title("Technology Evolution: F1 Score Progression", fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("results/comparison_evolution.png", bbox_inches='tight', dpi=150)
plt.show()
print("\n🎉 All visualizations saved to results/")


In [ ]:
# ── Final summary table ──────────────────────────────────────────
print("=" * 60)
print("FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"{'Model':20s} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 60)
for m_name in model_names:
    ov = all_results[m_name]['overall']
    print(f"{m_name:20s} {ov['precision']:10.4f} {ov['recall']:10.4f} {ov['f1']:10.4f}")
print("=" * 60)
print("\n📋 Key Findings:")
best = max(all_results, key=lambda x: all_results[x]['overall']['f1'])
print(f"  Best model  : {best} (F1={all_results[best]['overall']['f1']:.4f})")
print(f"  Improvement : Rule-Based → RoBERTa = "
      f"+{(all_results['RoBERTa']['overall']['f1'] - all_results['Rule-Based']['overall']['f1']):.4f} F1")
